In [20]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split
from math import sqrt

import warnings
warnings.filterwarnings('ignore')

In [21]:
data_ace_24 = pd.read_csv("../data/Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_24 = pd.read_csv("../data/Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_ace_af = pd.read_csv("../data/Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_af = pd.read_csv("../data/Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [22]:
class DANNRegressor:
    """
    Domain-Adversarial Neutral Network для адаптации discover -> ace
    fit(X_src, X_tgt), predict(X_src_test) -> X_tgt_hat
    """

    def __init__(self,
                 hidden_layer_size=64,
                 learning_rate=1e-3,
                 lambda_adapt=1.0,
                 maxiter=20,
                 adversarial_representation=True,
                 seed=42,
                 verbose=False):
        self.hidden_layer_size = hidden_layer_size
        self.learning_rate = learning_rate
        self.lambda_adapt = lambda_adapt if lambda_adapt not in (None, False) else 0.0
        self.maxiter = maxiter
        self.adversarial_representation = adversarial_representation
        self.seed = seed
        self.verbose = verbose

    def sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-z))

    def random_init(self, l_in, l_out):
        """ l_in - размер входных нейронов слоя, l_out - выходных """
        eps = sqrt(6.0 / (l_in + l_out))  # xavier initialization
        return eps * (2 * np.random.rand(l_out, l_in) - 1.0)

    def fit(self, X_src, X_tgt):
        """
        X_src: (N, D) данные discover
        X_tgt: (N, D) данные ace
        """
        N, D = X_src.shape
        K = D  # такая же размерность

        np.random.seed(self.seed)

        # вход -> скрытый слой
        W = self.random_init(D, self.hidden_layer_size)   # (H, D)
        b = np.zeros(self.hidden_layer_size)              # (H,)

        # скрытый слой -> выход (регрессия discover -> ace)
        V = self.random_init(self.hidden_layer_size, K)   # (K, H)
        c = np.zeros(K)                                   # (K,)

        # доменный классификатор по скрытому слою
        # random init, чтобы адаптация не спала в начале
        U = np.zeros(self.hidden_layer_size)              # (H,)
        d = 0.0                                           # скаляр

        for epoch in range(self.maxiter):
            for i in range(N):
                x_s = X_src[i, :]
                y_s = X_tgt[i, :]

                # регрессионная часть source -> target
                hidden_s = self.sigmoid(W @ x_s + b)   # (H,)
                y_pred = V @ hidden_s + c              # (K,)

                # задаем функцию потерь mse: L = 1/2 ||y_pred - y_s||^2
                err = y_pred - y_s                     # dL/dy_pred (K,)

                # градиенты по выходному слою (в исходном коде здесь softmax)
                delta_c = err                          # dL/dc
                delta_V = np.outer(err, hidden_s)      # dL/dV

                # dL/dh = V^T err, затем через сигмоид
                delta_hidden = V.T @ err               # dL/dh
                delta_hidden *= hidden_s * (1.0 - hidden_s)  # dL/dz
                delta_b = delta_hidden                 # dL/db
                delta_W = np.outer(delta_hidden, x_s)  # dL/dW

                # доменная часть
                if self.lambda_adapt != 0.0:
                    g_s = self.sigmoid(U @ hidden_s + d)   # доменная вероятность

                    # delta_d = λ (1 - g_s)
                    # delta_U = delta_d * hidden_s
                    delta_d = self.lambda_adapt * (1.0 - g_s)
                    delta_U = delta_d * hidden_s

                    if self.adversarial_representation:
                        # tmp = delta_d * U * h * (1 - h)
                        tmp = delta_d * U * hidden_s * (1.0 - hidden_s)
                        delta_b += tmp
                        delta_W += np.outer(tmp, x_s)

                    # добавляем регуляризатор от другого домена (target)
                    j = np.random.randint(N)
                    x_t = X_tgt[j, :]
                    hidden_t = self.sigmoid(W @ x_t + b)
                    g_t = self.sigmoid(U @ hidden_t + d)

                    delta_d -= self.lambda_adapt * g_t
                    delta_U -= self.lambda_adapt * g_t * hidden_t

                    if self.adversarial_representation:
                        tmp = -self.lambda_adapt * g_t * U * hidden_t * (1.0 - hidden_t)
                        delta_b += tmp
                        delta_W += np.outer(tmp, x_t)
                else:
                    delta_U = 0.0
                    delta_d = 0.0

                lr = self.learning_rate

                # минимизируем регрессионный loss + доменный регуляризатор
                W -= lr * delta_W
                b -= lr * delta_b
                V -= lr * delta_V
                c -= lr * delta_c

                # максимизирует расхождение
                U += lr * delta_U
                d += lr * delta_d

            if self.verbose:
                hidden_all = self.sigmoid(W @ X_src.T + b[:, None])     # (H, N)
                y_hat = (V @ hidden_all + c[:, None]).T                 # (N, K)
                mse = float(np.mean((y_hat - X_tgt) ** 2))
                print(f"Epoch {epoch+1}/{self.maxiter}, MSE source={mse:.4f}")

        self.W, self.V, self.b, self.c, self.U, self.d = W, V, b, c, U, d
        return self

    def _hidden(self, X):
        return self.sigmoid(self.W @ X.T + self.b[:, None])

    def predict(self, X_src):
        h = self._hidden(X_src)               # (H, N)
        Y_hat = self.V @ h + self.c[:, None]  # (K, N)
        return Y_hat.T                        # (N, K)

    def predict_domain(self, X):
        """0 - source, 1 - target (по скрытому слою)"""
        h = self._hidden(X)                   # (H, N)
        g = self.sigmoid(self.U @ h + self.d) # (N,)
        return (g < 0.5).astype(int)  #  <0.5 -> один домен, >0.5 -> другой


In [23]:
def adapt_discover_to_ace(ace_df, disc_df, future_lags, model, split_date="2022-01-01"):
    """
    Адаптация discover -> ace
    1. L обучается только на пересечении тренировочных данных ace и discover
    2. Тест discover адаптируется в домен данных ace
    3. Из всех данных возвращается только discover_test_adapted для дальнейшего предсказания
    """
    ace = ace_df.sort_index()
    disc = disc_df.sort_index()
    split_date = pd.Timestamp(split_date)

    ace_train = ace.loc[:split_date]
    disc_train = disc.loc[:split_date]
    disc_test = disc.loc[split_date:]   # адаптируем только тест discover

    # Пересечение индексов внутри train
    overlap_idx = ace_train.index.intersection(disc_train.index)

    ace_overlap = ace_train.loc[overlap_idx]
    disc_overlap = disc_train.loc[overlap_idx]

    feature_cols = [c for c in ace.columns if c not in future_lags]
    
    # Масштабирование
    sc_ace = StandardScaler().fit(ace_overlap[feature_cols])
    sc_disc = StandardScaler().fit(disc_overlap[feature_cols])

    X_ace = sc_ace.transform(ace_overlap[feature_cols])
    X_disc = sc_disc.transform(disc_overlap[feature_cols])
    
    L = model
    L.fit(X_disc, X_ace)

    X_disc_test = sc_disc.transform(disc_test[feature_cols])            # Нормировка тестового набора данных discover
    X_disc_test_adapted = L.predict(X_disc_test)                        # Применение обученной модели адаптации
    X_disc_test_adapted = sc_ace.inverse_transform(X_disc_test_adapted) # Приводим новые адаптированные данные к ненормированному виду

    disc_test_adapted = pd.DataFrame(X_disc_test_adapted, index=disc_test.index, columns=feature_cols)

    # Целевые переменные возвращаются обратно неизменёнными
    for col in future_lags:
        disc_test_adapted[col] = disc_test[col]

    return disc_test_adapted, L, sc_disc, sc_ace

In [24]:
def build_models(random_state=42):
    models = {}
    models['Linear'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
    
    models['Ridge'] = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=1.0,
            solver='auto',
            random_state=random_state))
    ])

    models['Lasso'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(
            alpha=0.0005,
            tol=0.01,
            max_iter=500, 
            random_state=random_state))
    ])
    
    models['LGBM'] = Pipeline([
        ("boost", LGBMRegressor(
            boosting_type='gbdt',
            num_leaves=93,
            max_depth=5,
            learning_rate=0.014357868416776678,
            n_estimators=1160,
            # subsample_for_bin=200000,
            objective=None,
            class_weight=None,
            min_split_gain=0.0,
            min_child_weight=0.001,
            min_child_samples=25,
            subsample=0.5606239658637814,
            subsample_freq=0,
            colsample_bytree=0.9879825765791656,
            reg_alpha=0.022687699993507067,
            reg_lambda=0.005915305250069859,
            random_state=random_state,
            n_jobs=None,
            importance_type='split',
            metric='rmse',
            # early_stopping_rounds=50,
            verbose=-1))
    ])
    
    models['MLP'] = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(12,),
            solver='adam',
            alpha=1e-4,
            batch_size=32,
            learning_rate_init=1e-3,
            early_stopping=True,
            validation_fraction=0.1,
            max_iter=1500,
            random_state=random_state))
    ])
    return models
    
models = build_models()

def evaluate_M_A(ace_df, disc_test_adapted_df, split_date, target, future_lags, adaptation_method='linreg', delays='24h', results_list=None):
    """
    Модель M_A обучается на ace_train и тестируется на адаптированном discover_test
    """
    split_date = pd.Timestamp(split_date)

    ace_train = ace_df.loc[:split_date]

    feature_cols = [c for c in ace_train.columns if c not in future_lags]

    X_train = ace_train[feature_cols].values
    y_train = ace_train[target].values

    X_test = disc_test_adapted_df[feature_cols].values
    y_test = disc_test_adapted_df[target].values

    for name, model in models.items():
        if name == 'LGBM':
            # Для бустинга выделяем валидационный набор
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_train, y_train, test_size=0.1, random_state=42)

            model.fit(X_tr, y_tr,
                boost__eval_set=[(X_val, y_val)],
                boost__eval_metric='l2')
        else:
            model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        
        if results_list is None:
            results_list = []
            
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results_list.append({
            'target': target,
            'delays': delays,
            'adaptation_method': adaptation_method,
            'forecast_model': name,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2
        })

        print(f"{name}: rmse={rmse:.4f}, mae={mae:.4f}, r2={r2:.4f}")

    return results_list

In [25]:
def evaluate_M_A_check_params(ace_df, disc_test_adapted_df, split_date, target, future_lags, adaptation_method='linreg', delays='24h'):

    split_date = pd.Timestamp(split_date)

    ace_train = ace_df.loc[:split_date]

    feature_cols = [c for c in ace_train.columns if c not in future_lags]

    X_train = ace_train[feature_cols].values
    y_train = ace_train[target].values

    X_test = disc_test_adapted_df[feature_cols].values
    y_test = disc_test_adapted_df[target].values

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
            
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
        
    # rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    # mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return r2

In [26]:
# 0. Копирование загруженных данных в отдельные переменные
data_ace_24_copy = data_ace_24.copy()
data_discover_24_copy = data_discover_24.copy()
data_ace_af_copy = data_ace_af.copy()
data_discover_af_copy = data_discover_af.copy()

# 1. Задание переменных для адаптации
split_date = "2024-01-01"
future_lags = [f'Dst_plus{i}' for i in range(1, 25)] # все сдвиги во времени вперед (задаются при обработке данных)
targets = ['Dst_plus1', 'Dst_plus3', 'Dst_plus6'] # таргеты, на которые делаем прогнозы

In [9]:
# trial_ids = []
# r2s = []
# params = []

# target = 'Dst_plus1'

# n_trials = 30

# for t in range(n_trials):
#     # случайный набор гиперпараметров DANN
#     hidden = np.random.choice([512, 1024, 1536])
#     lr = np.random.uniform(1e-5, 1e-3)
#     lam = np.random.uniform(0.5, 2.0)
#     maxiter = np.random.choice([20, 30, 40])

#     print(hidden, lr, lam, maxiter)
    
#     model = DANNRegressor(
#         hidden_layer_size=hidden,
#         learning_rate=lr,
#         lambda_adapt=lam,
#         maxiter=maxiter,
#         verbose=False
#     )

#     disc_test_adapted = adapt_discover_to_ace(
#         data_ace_24_copy,
#         data_discover_24_copy,
#         future_lags,
#         model,
#         split_date
#     )

#     disc_test_adapted_df = disc_test_adapted[0]

#     r2 = evaluate_M_A_check_params(
#         data_ace_24_copy,
#         disc_test_adapted_df,
#         split_date,
#         target,
#         future_lags
#     )

#     print(r2)

#     trial_ids.append(t)
#     r2s.append(r2)
#     params.append(dict(
#         hidden_layer_size=hidden,
#         learning_rate=lr,
#         lambda_adapt=lam,
#         maxiter=maxiter
#     ))

# best_i = int(np.argmax(r2s))
# best_r2 = r2s[best_i]
# best_params = params[best_i]

# print("best r2:", best_r2)
# print("best params:", best_params)

In [24]:
# target = 'Dst_plus1'

# # модель полегче для более быстрого теста
# hidden = 128
# lr = 2e-4
# maxiter = 10

# # lam_params = [0.8, 0.9, 1, 1.2, 1.5, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 50]
# lr_params = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2] 

# for lr in lr_params:
# # for lam in lam_params:
#     # print(lam)
#     print(lr)
    
#     model = DANNRegressor(
#         hidden_layer_size=hidden,
#         learning_rate=lr,
#         lambda_adapt=lam,
#         maxiter=maxiter,
#         verbose=False
#     )

#     disc_test_adapted = adapt_discover_to_ace(
#         data_ace_24_copy,
#         data_discover_24_copy,
#         future_lags,
#         model,
#         split_date
#     )

#     disc_test_adapted_df = disc_test_adapted[0]

#     r2 = evaluate_M_A_check_params(
#         data_ace_24_copy,
#         disc_test_adapted_df,
#         split_date,
#         target,
#         future_lags
#     )
#     print(r2)

1e-06
0.3933155889202894
1e-05
0.6521359735031269
0.0001
0.7995633519789317
0.001
0.7565681892282791
0.01
0.3178175044475732


In [27]:
# model = DANNRegressor(
#     hidden_layer_size=2048,
#     learning_rate=2e-4,
#     lambda_adapt=6,
#     maxiter=40,
#     verbose=True
# )

model = DANNRegressor(
    hidden_layer_size=256,
    learning_rate=4e-4,
    lambda_adapt=2,
    maxiter=30,
    verbose=True
)

In [18]:
disc_adapted_dann_d_to_a_24, L_dann_24, sd_dann_24, sa_dann_24 = adapt_discover_to_ace(
    data_ace_24_copy,
    data_discover_24_copy,
    future_lags,
    model,
    split_date
)

Epoch 1/30, MSE source=0.2995
Epoch 2/30, MSE source=0.2868
Epoch 3/30, MSE source=0.2939
Epoch 4/30, MSE source=0.2981
Epoch 5/30, MSE source=0.2991
Epoch 6/30, MSE source=0.2975
Epoch 7/30, MSE source=0.2903
Epoch 8/30, MSE source=0.2821
Epoch 9/30, MSE source=0.2743
Epoch 10/30, MSE source=0.2697
Epoch 11/30, MSE source=0.2654
Epoch 12/30, MSE source=0.2624
Epoch 13/30, MSE source=0.2597
Epoch 14/30, MSE source=0.2575
Epoch 15/30, MSE source=0.2563
Epoch 16/30, MSE source=0.2548
Epoch 17/30, MSE source=0.2535
Epoch 18/30, MSE source=0.2518
Epoch 19/30, MSE source=0.2509
Epoch 20/30, MSE source=0.2491
Epoch 21/30, MSE source=0.2481
Epoch 22/30, MSE source=0.2473
Epoch 23/30, MSE source=0.2463
Epoch 24/30, MSE source=0.2449
Epoch 25/30, MSE source=0.2437
Epoch 26/30, MSE source=0.2425
Epoch 27/30, MSE source=0.2409
Epoch 28/30, MSE source=0.2399
Epoch 29/30, MSE source=0.2393
Epoch 30/30, MSE source=0.2389


In [28]:
disc_adapted_dann_d_to_a_af, L_dann_af, sd_dann_af, sa_dann_af = adapt_discover_to_ace(
    data_ace_af_copy,
    data_discover_af_copy,
    future_lags,
    model,
    split_date
)

Epoch 1/30, MSE source=0.2857
Epoch 2/30, MSE source=0.2733
Epoch 3/30, MSE source=0.2790
Epoch 4/30, MSE source=0.2842
Epoch 5/30, MSE source=0.2802
Epoch 6/30, MSE source=0.2694
Epoch 7/30, MSE source=0.2621
Epoch 8/30, MSE source=0.2580
Epoch 9/30, MSE source=0.2565
Epoch 10/30, MSE source=0.2570
Epoch 11/30, MSE source=0.2582
Epoch 12/30, MSE source=0.2605
Epoch 13/30, MSE source=0.2604
Epoch 14/30, MSE source=0.2593
Epoch 15/30, MSE source=0.2588
Epoch 16/30, MSE source=0.2593
Epoch 17/30, MSE source=0.2615
Epoch 18/30, MSE source=0.2610
Epoch 19/30, MSE source=0.2637
Epoch 20/30, MSE source=0.2667
Epoch 21/30, MSE source=0.2713
Epoch 22/30, MSE source=0.2775
Epoch 23/30, MSE source=0.2819
Epoch 24/30, MSE source=0.2818
Epoch 25/30, MSE source=0.2772
Epoch 26/30, MSE source=0.2668
Epoch 27/30, MSE source=0.2549
Epoch 28/30, MSE source=0.2434
Epoch 29/30, MSE source=0.2342
Epoch 30/30, MSE source=0.2289


In [33]:
results_list = []

In [19]:
print(f"\n==== Depth - 24h ====")

print(f"\n==== Adaptation - discover-to-ace DANN ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_A(data_ace_24_copy,
                       disc_adapted_dann_d_to_a_24,
                       split_date,
                       target_col,
                       future_lags,
                       adaptation_method='disc-ace-dann',
                       delays='24h',
                       results_list=results_list)


==== Depth - 24h ====

==== Adaptation - discover-to-ace DANN ====

==== Forecast of DST_PLUS1 ====
Linear: rmse=9.9459, mae=3.9227, r2=0.8198
Ridge: rmse=9.9459, mae=3.9228, r2=0.8198
Lasso: rmse=9.9523, mae=3.9263, r2=0.8196
LGBM: rmse=9.8221, mae=3.8970, r2=0.8243
MLP: rmse=9.8952, mae=3.8804, r2=0.8217

==== Forecast of DST_PLUS3 ====
Linear: rmse=11.8139, mae=5.9893, r2=0.7458
Ridge: rmse=11.8139, mae=5.9893, r2=0.7458
Lasso: rmse=11.8128, mae=5.9877, r2=0.7459
LGBM: rmse=11.4437, mae=5.7941, r2=0.7615
MLP: rmse=11.6568, mae=5.8674, r2=0.7525

==== Forecast of DST_PLUS6 ====
Linear: rmse=14.2822, mae=7.9769, r2=0.6284
Ridge: rmse=14.2822, mae=7.9769, r2=0.6284
Lasso: rmse=14.2822, mae=7.9751, r2=0.6284
LGBM: rmse=13.9517, mae=7.9302, r2=0.6454
MLP: rmse=14.3015, mae=8.2664, r2=0.6274


In [34]:
print(f"\n==== Depth - autocorrelation function ====")

print(f"\n==== Adaptation - discover-to-ace DANN ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_A(data_ace_af_copy,
                       disc_adapted_dann_d_to_a_af,
                       split_date,
                       target_col,
                       future_lags,
                       adaptation_method='disc-ace-dann',
                       delays='auto_func',
                       results_list=results_list)


==== Depth - autocorrelation function ====

==== Adaptation - discover-to-ace DANN ====

==== Forecast of DST_PLUS1 ====
Linear: rmse=13.0208, mae=5.1189, r2=0.8005
Ridge: rmse=13.0209, mae=5.1190, r2=0.8005
Lasso: rmse=13.0254, mae=5.1265, r2=0.8004
LGBM: rmse=12.8908, mae=5.0421, r2=0.8045
MLP: rmse=12.7286, mae=4.9942, r2=0.8094

==== Forecast of DST_PLUS3 ====
Linear: rmse=15.2392, mae=7.3474, r2=0.7269
Ridge: rmse=15.2393, mae=7.3475, r2=0.7269
Lasso: rmse=15.2414, mae=7.3486, r2=0.7268
LGBM: rmse=14.8067, mae=7.0464, r2=0.7422
MLP: rmse=14.8276, mae=7.1060, r2=0.7415

==== Forecast of DST_PLUS6 ====
Linear: rmse=18.2064, mae=9.4983, r2=0.6104
Ridge: rmse=18.2065, mae=9.4983, r2=0.6104
Lasso: rmse=18.2083, mae=9.4986, r2=0.6103
LGBM: rmse=17.8007, mae=9.2164, r2=0.6275
MLP: rmse=17.8942, mae=9.3806, r2=0.6236


In [35]:
results_df = pd.DataFrame(results_list)

In [36]:
results_df

,target,delays,adaptation_method,forecast_model,RMSE,MAE,R2
0,Dst_plus1,auto_func,disc-ace-dann,Linear,13.020832,5.118924,0.800540
1,Dst_plus1,auto_func,disc-ace-dann,Ridge,13.020865,5.119005,0.800539
2,Dst_plus1,auto_func,disc-ace-dann,Lasso,13.025404,5.126530,0.800400
3,Dst_plus1,auto_func,disc-ace-dann,LGBM,12.890807,5.042106,0.804503
4,Dst_plus1,auto_func,disc-ace-dann,MLP,12.728646,4.994204,0.809391
5,Dst_plus3,auto_func,disc-ace-dann,Linear,15.239227,7.347405,0.726903
6,Dst_plus3,auto_func,disc-ace-dann,Ridge,15.239289,7.347462,0.726901
7,Dst_plus3,auto_func,disc-ace-dann,Lasso,15.241391,7.348580,0.726826
8,Dst_plus3,auto_func,disc-ace-dann,LGBM,14.806722,7.046384,0.742185
9,Dst_plus3,auto_func,disc-ace-dann,MLP,14.827644,7.105990,0.741456


In [37]:
results_df.to_excel("../results/models_adaptation_discover_to_ace_dann_AF_2024-2026.xlsx", index=False)